In [2]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import r2_score, mean_absolute_error
from sklearn.model_selection import train_test_split


# ===== Загрузка =====
df = pd.read_csv("train_valid.csv")


In [3]:
# ==================== ПОДГОТОВКА ДАННЫХ ДЛЯ ОБУЧЕНИЯ ====================
# Создаем копию данных
df_processed_train = df.copy()

# НОВЫЕ ПРИЗНАКИ: Взаимодействия (добавляем в самом начале)
df_processed_train['Rooms_Square_Ratio'] = df_processed_train['Rooms'] / df_processed_train['Square']
df_processed_train['LifeSquare_Ratio'] = df_processed_train['LifeSquare'] / df_processed_train['Square']

# На тренировочных данных вычисляем средние
target_encoding = df_processed_train.groupby('DistrictId')['Price'].mean()
price_per_sqm_by_district = df_processed_train.groupby('DistrictId')['Price'].mean() / df_processed_train.groupby('DistrictId')['Square'].mean()
room_means = df_processed_train.groupby(['DistrictId', 'Rooms'])['Price'].mean().reset_index()
room_means.rename(columns={'Price': 'Mean_Price_By_Rooms'}, inplace=True)

# НОВЫЙ ПРИЗНАК: Средняя цена за кв.м жилой площади по району и количеству комнат
life_square_means = df_processed_train.groupby(['DistrictId', 'Rooms']).apply(
    lambda x: x['Price'].mean() / x['LifeSquare'].mean() if x['LifeSquare'].mean() > 0 else 0
).reset_index()
life_square_means.rename(columns={0: 'Mean_Price_By_LifeSquare'}, inplace=True)

# НОВЫЙ ПРИЗНАК: Средняя цена за кв.м общей площади по району и количеству комнат
square_means = df_processed_train.groupby(['DistrictId', 'Rooms']).apply(
    lambda x: x['Price'].mean() / x['Square'].mean() if x['Square'].mean() > 0 else 0
).reset_index()
square_means.rename(columns={0: 'Mean_Price_By_Square'}, inplace=True)

train_target_encoding = target_encoding
train_price_per_sqm = price_per_sqm_by_district
train_room_means = room_means
train_life_square_means = life_square_means
train_square_means = square_means
train_district_means = df_processed_train.groupby('DistrictId')['Price'].mean()
train_overall_mean_price = df_processed_train['Price'].mean()

# Сохраняем средние для новых признаков взаимодействия
train_rooms_square_ratio_mean = df_processed_train['Rooms_Square_Ratio'].mean()
train_life_square_ratio_mean = df_processed_train['LifeSquare_Ratio'].mean()

# Создаем признаки
df_processed_train['DistrictId_TargetEnc'] = df_processed_train['DistrictId'].map(target_encoding)
df_processed_train['Avg_Price_Per_Sqm_By_District'] = df_processed_train['DistrictId'].map(price_per_sqm_by_district)

# Объединяем средние по комнатам
df_processed_train = df_processed_train.merge(room_means, on=['DistrictId', 'Rooms'], how='left')

# НОВЫЙ ПРИЗНАК: Объединяем средние по жилой площади
df_processed_train = df_processed_train.merge(life_square_means, on=['DistrictId', 'Rooms'], how='left')

# НОВЫЙ ПРИЗНАК: Объединяем средние по общей площади
df_processed_train = df_processed_train.merge(square_means, on=['DistrictId', 'Rooms'], how='left')

# Заполняем пропуски в новых признаках
overall_life_square_price = df_processed_train['Price'].mean() / df_processed_train['LifeSquare'].mean() if df_processed_train['LifeSquare'].mean() > 0 else 0
overall_square_price = df_processed_train['Price'].mean() / df_processed_train['Square'].mean() if df_processed_train['Square'].mean() > 0 else 0

if 'Mean_Price_By_LifeSquare' in df_processed_train.columns:
    df_processed_train['Mean_Price_By_LifeSquare'] = df_processed_train['Mean_Price_By_LifeSquare'].fillna(overall_life_square_price)
else:
    df_processed_train['Mean_Price_By_LifeSquare'] = overall_life_square_price

if 'Mean_Price_By_Square' in df_processed_train.columns:
    df_processed_train['Mean_Price_By_Square'] = df_processed_train['Mean_Price_By_Square'].fillna(overall_square_price)
else:
    df_processed_train['Mean_Price_By_Square'] = overall_square_price

# Заполняем пропуски в Mean_Price_By_Rooms
df_processed_train['Mean_Price_By_Rooms'] = df_processed_train['Mean_Price_By_Rooms'].fillna(
    df_processed_train['DistrictId'].map(train_district_means)
)

# Удаляем столбцы
columns_to_drop = ['Id', 'Healthcare_1','Ecology_1']
df_processed_train = df_processed_train.drop(columns=columns_to_drop)

# Подготовка категориальных признаков
categorical_columns = df_processed_train.select_dtypes(include=['object']).columns
for col in categorical_columns:
    df_processed_train[col] = df_processed_train[col].astype(str)

print("✅ ПРИЗНАКИ ДЛЯ ОБУЧЕНИЯ ПОДГОТОВЛЕНЫ")
print(f"Добавлены новые признаки:")
print(f"DistrictId_TargetEnc - средняя цена квартир в каждом районе (таргет-энкодинг по DistrictId)")
print(f"Mean_Price_By_Rooms - средняя цена квартир по комбинации 'район + количество комнат'")
print(f"Mean_Price_By_Square - средняя цена за кв.м общей площади по 'район + комнаты'")
print(f"Mean_Price_By_LifeSquare - средняя цена за кв.м ЖИЛОЙ площади по 'район + комнаты'")
print(f"Avg_Price_Per_Sqm_By_District - средняя цена за кв.м по районам (без учета комнат)")
print(f"LifeSquare_Ratio - доля жилой площади от общей (эффективность планировки)")
print(f"Rooms_Square_Ratio - плотность комнат (количество комнат на кв.метр)")


✅ ПРИЗНАКИ ДЛЯ ОБУЧЕНИЯ ПОДГОТОВЛЕНЫ
Добавлены новые признаки:
DistrictId_TargetEnc - средняя цена квартир в каждом районе (таргет-энкодинг по DistrictId)
Mean_Price_By_Rooms - средняя цена квартир по комбинации 'район + количество комнат'
Mean_Price_By_Square - средняя цена за кв.м общей площади по 'район + комнаты'
Mean_Price_By_LifeSquare - средняя цена за кв.м ЖИЛОЙ площади по 'район + комнаты'
Avg_Price_Per_Sqm_By_District - средняя цена за кв.м по районам (без учета комнат)
LifeSquare_Ratio - доля жилой площади от общей (эффективность планировки)
Rooms_Square_Ratio - плотность комнат (количество комнат на кв.метр)


C:\Users\Админ\AppData\Local\Temp\ipykernel_7852\2346112067.py:16: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  life_square_means = df_processed_train.groupby(['DistrictId', 'Rooms']).apply(
C:\Users\Админ\AppData\Local\Temp\ipykernel_7852\2346112067.py:22: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  square_means = df_processed_train.groupby(['DistrictId', 'Rooms']).apply(


In [4]:


# Определение фичей и таргета
target = 'Price'
features = [col for col in df_processed_train.columns if col != target]

# Разделение на train/valid
X_train, X_valid, y_train, y_valid = train_test_split(
    df_processed_train[features], df_processed_train[target],
    test_size=0.2, random_state=42
)

# Параметры для поиска
param_grid = {
    'n_estimators': [200, 300, 400],
    'max_depth': [15, 20, None],
    'min_samples_split': [2, 5],
    'min_samples_leaf': [1, 2],
    'max_features': ['sqrt', 'log2']
}

# Поиск лучших параметров
rf = RandomForestRegressor(random_state=42, n_jobs=-1)
grid_search = GridSearchCV(
    estimator=rf,
    param_grid=param_grid,
    cv=3,
    scoring='neg_mean_absolute_error',
    n_jobs=-1,
    verbose=1
)

grid_search.fit(X_train, y_train)

# Лучшие параметры
print("Лучшие параметры:", grid_search.best_params_)
best_model = grid_search.best_estimator_

Fitting 3 folds for each of 72 candidates, totalling 216 fits
Лучшие параметры: {'max_depth': 20, 'max_features': 'sqrt', 'min_samples_leaf': 1, 'min_samples_split': 2, 'n_estimators': 400}


In [5]:
# Определение фичей и таргета
target = 'Price'
features = [col for col in df_processed_train.columns if col != target]

# Разделение на train/valid
X_train, X_valid, y_train, y_valid = train_test_split(
    df_processed_train[features], df_processed_train[target],
    test_size=0.2, random_state=42
)

model = RandomForestRegressor(
    n_estimators=400,           # Увеличил количество деревьев
    max_depth=20,               # Ограничил глубину для борьбы с переобучением
    min_samples_split=2,        # Увеличил для лучшей обобщающей способности
    min_samples_leaf=1,         # Оставил оптимальным
    max_features='sqrt',        # Лучше чем 'auto' для регрессии
    bootstrap=True,
    oob_score=True,             # Включил out-of-bag оценку
    random_state=42,
    n_jobs=-1,
    verbose=1
)

print("Начинаем обучение RandomForest...")
model.fit(X_train, y_train)

# Out-of-bag оценка
print(f"OOB Score: {model.oob_score_:.4f}")

Начинаем обучение RandomForest...


[Parallel(n_jobs=-1)]: Using backend ThreadingBackend with 12 concurrent workers.
[Parallel(n_jobs=-1)]: Done  26 tasks      | elapsed:    0.0s
[Parallel(n_jobs=-1)]: Done 176 tasks      | elapsed:    0.3s
[Parallel(n_jobs=-1)]: Done 400 out of 400 | elapsed:    0.8s finished


OOB Score: 0.7818


In [6]:
preds = model.predict(X_valid)

[Parallel(n_jobs=12)]: Using backend ThreadingBackend with 12 concurrent workers.
[Parallel(n_jobs=12)]: Done  26 tasks      | elapsed:    0.0s
[Parallel(n_jobs=12)]: Done 176 tasks      | elapsed:    0.0s
[Parallel(n_jobs=12)]: Done 400 out of 400 | elapsed:    0.1s finished


In [7]:
# Метрики
print("\n" + "="*50)
print("РЕЗУЛЬТАТЫ ОБУЧЕНИЯ:")
print("="*50)
print(f"MAE: {mean_absolute_error(y_valid, preds):.2f}")
print(f"R2: {r2_score(y_valid, preds):.4f}")
print("="*50)


РЕЗУЛЬТАТЫ ОБУЧЕНИЯ:
MAE: 27883.46
R2: 0.7426
